[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/phase_change/phase_change.ipynb)

# Boiling at Fixed Pressure

This example computes phase coexistence and isobaric boiling of a van der Waals fluid. Steady heating under a fixed pressure warms the liquid to the saturation temperature, converts it into vapour at constant temperature and pressure, and then warms the vapour. Below the critical point, liquid and vapour coexist at the two volumes where their pressures and chemical potentials match, and boiling absorbs the enthalpy jump between them, the latent heat.

Two degenerate quantity directions `l0 g0` form an exterior state space for the liquid and vapour volumes. A coexistence state is modelled as an antivector combining `l.dual()` and `g.dual()`, read back without a metric by the regressive pairings `l & state` and `g & state`. Evaluating the pressure and chemical-potential differences maps each trial state back to the same state space; because the isothermal slope of chemical potential is volume times the slope of pressure, the Jacobian of that residual is a two-dyad linear map on state variations that `.solve()` inverts in each Newton step across a batch of temperatures.

In classical thermodynamics notation, Maxwell's construction equates $p(V_l) = p(V_g)$ and $\mu(V_l) = \mu(V_g)$ with $\mu = F + pV$, cutting the isotherm loop at $p_\text{sat}$ into equal areas via $\int_{V_l}^{V_g} p\,dV = p_\text{sat}(V_g - V_l)$. Isobaric heating satisfies $dQ = dH$, with latent heat $L = H_g - H_l = T(S_g - S_l)$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.util.animation import show_states
from examples.thermodynamics.phase_change import render
from examples.thermodynamics.phase_change.scenarios import stage_frames

np.set_printoptions(precision=4, suppress=True)

ga = Algebra("l0g0")
mv = NumpyContext(ga, np.float64).multivector
Scalar = ga.gatype.scalar()                                # Scalar+
State = ga.gatype.antivector()                             # Antivector-
Jacobian = ga.gatype((State, State))                       # State- <- State-

# Regressive pairings read the liquid and vapour volumes without a metric.
liquid_volume = mv.l & State                               # [] Scalar+ <- State-
vapour_volume = mv.g & State                               # [] Scalar+ <- State-

# Reduced units measure temperature and volume in their critical values, energy in the gas constant times
# the critical temperature, and pressure in that energy per critical volume, giving critical pressure 3/8.
capacity, attraction, exclusion = 1.5, 9 / 8, 1 / 3

## Liquid and vapour in coexistence

Above the critical temperature, pressure falls steadily as volume grows; below it, each isotherm develops an unstable loop across which the fluid separates into liquid and vapour. Because pressure is minus the volume slope of the Helmholtz free energy and the chemical potential is that tangent line's intercept at zero volume, matching both potentials places the two phases on a single common tangent to the free-energy curve.

In [ ]:
def pressure(volume: Scalar, temperature: np.ndarray | float) -> Scalar:
    """The van der Waals equation of state balances thermal pressure in the free volume against molecular attraction."""
    return temperature / (volume - exclusion) - attraction / volume.squared()


def pressure_slope(volume: Scalar, temperature: np.ndarray | float) -> Scalar:
    """At fixed temperature, pressure changes with volume from free-volume compression and molecular attraction."""
    return -temperature / (volume - exclusion).squared() + 2 * attraction / volume ** 3


def helmholtz_energy(volume: Scalar, temperature: np.ndarray | float) -> Scalar:
    """The Helmholtz free energy has minus the pressure as its slope with respect to volume at fixed temperature."""
    return (-capacity * temperature * np.log(temperature) - temperature * (volume - exclusion).log()
            - attraction / volume)


def chemical_potential(volume: Scalar, temperature: np.ndarray | float) -> Scalar:
    """The chemical potential adds pressure-volume work to the Helmholtz free energy, giving the free-energy tangent's
    intercept at zero volume."""
    return helmholtz_energy(volume, temperature) + pressure(volume, temperature) * volume


def enthalpy(volume: Scalar, temperature: np.ndarray | float) -> Scalar:
    """The enthalpy adds pressure-volume work to the internal energy, so its change at fixed pressure equals the heat
    absorbed."""
    return capacity * temperature - attraction / volume + pressure(volume, temperature) * volume


def isobar_temperature(volume: Scalar, isobar_pressure: Scalar) -> Scalar:
    """Along an isobar of fixed pressure, the equation of state gives the temperature at each volume."""
    return (isobar_pressure + attraction / volume.squared()) * (volume - exclusion)

In [ ]:
n_newton = 8
boiling_temperature = 0.85


def mismatch(state: State, temperature: np.ndarray | float) -> State:
    """Two trial volumes fail to coexist by their pressure difference along the liquid direction and their
    chemical-potential difference along the vapour direction."""
    liquid, vapour = liquid_volume(state), vapour_volume(state)
    pressure_gap = pressure(liquid, temperature) - pressure(vapour, temperature)
    chemical_gap = chemical_potential(liquid, temperature) - chemical_potential(vapour, temperature)
    return mv.l.dual() * pressure_gap + mv.g.dual() * chemical_gap


def mismatch_jacobian(state: State, temperature: np.ndarray | float) -> Jacobian:
    """By the Gibbs-Duhem relation a phase's isothermal chemical-potential slope is volume times its pressure slope,
    forming a two-dyad map from volume variations to pressure and chemical-potential changes."""
    liquid, vapour = liquid_volume(state), vapour_volume(state)
    liquid_step = (mv.l.dual() + liquid * mv.g.dual()) * pressure_slope(liquid, temperature)
    vapour_step = (mv.l.dual() + vapour * mv.g.dual()) * pressure_slope(vapour, temperature)
    return liquid_step * liquid_volume - vapour_step * vapour_volume                     # [...] State- <- State-


def coexistence(temperature: np.ndarray | float) -> State:
    """Liquid and vapour coexist at each temperature at the two volumes where both pressure and chemical potential
    match."""
    # Below the critical point the coexisting volumes part with the square root of the temperature drop,
    # starting the liquid branch below the critical volume and the vapour branch above it.
    departure = np.sqrt(1 - temperature)
    state = mv.l.dual() / (1 + 2 * departure) + mv.g.dual() / (1 - departure) ** 3       # [...] State-
    for _ in range(n_newton):
        state = state - mismatch_jacobian(state, temperature).solve(mismatch(state, temperature))
    return state


render.draw_charts(pressure, enthalpy, coexistence, isobar_temperature, boiling_temperature);

On the left, the pressure–volume chart shows red isotherms beneath the black coexistence dome, with liquid volumes rising along the left branch to the critical point and vapour volumes descending along the right branch. At the boiling temperature, the bold isotherm crosses its loop at the saturation pressure, enclosing equal blue and orange areas below and above the tie line. In the middle, the temperature–enthalpy chart shows purple isobars that rise through the liquid, cross the dome horizontally during boiling, and rise again through the vapour, crossed by grey-blue curves of constant vapour fraction. On the right, the phase diagram shows the saturation curve ending at the critical point alongside green lines of constant volume.

## Heating at fixed pressure

A loaded piston fixes the pressure at the saturation pressure of the boiling temperature while heat enters at a steady rate. Because heat added at constant pressure equals the change in enthalpy, the fluid advances along its isobar in equal enthalpy steps. The liquid warms to the boiling temperature, converts into vapour at constant temperature with its volume and enthalpy weighted by the vapour fraction, and then warms further as vapour.

This cell and the next take about ten seconds, most of it drawing the animation.

In [ ]:
cold_temperature, hot_temperature = 0.7, 0.98
n_frames, frame_duration_ms = 105, 40


def volume_at(temperature: np.ndarray | float, isobar_pressure: Scalar, guess: Scalar) -> Scalar:
    """At fixed pressure, each temperature corresponds to a volume on the liquid or vapour branch of the isobar."""
    volume = guess
    for _ in range(n_newton):
        gap = isobar_temperature(volume, isobar_pressure) - temperature
        slope = isobar_pressure - attraction * (volume - 2 * exclusion) / volume ** 3
        volume = volume - gap / slope                                                    # [...] Scalar+
    return volume


def mixture(state: State, vapour_fraction: float, temperature: float) -> tuple[Scalar, Scalar, Scalar]:
    """A two-phase mixture shares volume and enthalpy linearly between liquid and vapour by the vapour fraction."""
    liquid, vapour = liquid_volume(state), vapour_volume(state)
    volume = (1 - vapour_fraction) * liquid + vapour_fraction * vapour
    mixture_enthalpy = ((1 - vapour_fraction) * enthalpy(liquid, temperature)
                        + vapour_fraction * enthalpy(vapour, temperature))
    return volume, mixture_enthalpy, (1 - vapour_fraction) * liquid

In [ ]:
saturation_state = coexistence(boiling_temperature)                                  # [] State-
liquid, vapour = liquid_volume(saturation_state), vapour_volume(saturation_state)    # [] Scalar+, Scalar+
saturation_pressure = pressure(liquid, boiling_temperature)                          # [] Scalar+


def heating():
    """Steady heat input advances the fluid in equal enthalpy steps from cold liquid through boiling to hot vapour."""
    # Each stage receives frames in proportion to the enthalpy jump between its endpoints.
    stage_enthalpies = stack([
        enthalpy(volume_at(cold_temperature, saturation_pressure, liquid), cold_temperature),
        enthalpy(liquid, boiling_temperature),
        enthalpy(vapour, boiling_temperature),
        enthalpy(volume_at(hot_temperature, saturation_pressure, vapour), hot_temperature),
    ])                                                                               # [4] Scalar+
    counts = stage_frames(stage_enthalpies, n_frames)
    for temperature in np.linspace(cold_temperature, boiling_temperature, counts[0], endpoint=False):
        volume = volume_at(temperature, saturation_pressure, liquid)
        yield volume, mv.scalar([temperature]), enthalpy(volume, temperature), volume
    for vapour_fraction in np.linspace(0, 1, counts[1], endpoint=False):
        volume, mixture_enthalpy, liquid_share = mixture(saturation_state, vapour_fraction, boiling_temperature)
        yield volume, mv.scalar([boiling_temperature]), mixture_enthalpy, liquid_share
    for temperature in np.linspace(boiling_temperature, hot_temperature, counts[2]):
        volume = volume_at(temperature, saturation_pressure, vapour)
        yield volume, mv.scalar([temperature]), enthalpy(volume, temperature), volume * 0


stages = show_states(
    heating(), render.heating, pressure, enthalpy, coexistence, isobar_temperature, boiling_temperature,
    name="phase_change", duration_ms=frame_duration_ms, n_frames=n_frames,
)

In the cylinder, the liquid at the bottom warms with very little expansion until it reaches the boiling temperature. Vapour then forms above the liquid and raises the loaded piston at constant temperature while the state crosses the tie line on the pressure–volume chart, traverses the flat plateau of the bold isobar on the temperature–enthalpy chart, and pauses on the saturation curve in the phase diagram. Once all the liquid has boiled away, the vapour warms and expands rapidly. Because the enthalpy plateau spans the latent heat, steady heating spends most of its duration boiling the fluid.

In [ ]:
latent_heat = enthalpy(vapour, boiling_temperature) - enthalpy(liquid, boiling_temperature)   # [] Scalar+
print("coexisting volumes:", liquid.to_array(), vapour.to_array())
print("saturation pressure:", saturation_pressure.to_array())
print("latent heat:", latent_heat.to_array())

In [ ]:
# checks
residual = mismatch(saturation_state, boiling_temperature)
np.testing.assert_allclose(liquid_volume(residual).to_array(), 0, atol=1e-10)
np.testing.assert_allclose(vapour_volume(residual).to_array(), 0, atol=1e-10)
# Latent heat equals the boiling temperature times the entropy jump across the free-volume ratio.
entropy_jump = ((vapour - exclusion) / (liquid - exclusion)).log()
np.testing.assert_allclose(latent_heat.to_array(), (boiling_temperature * entropy_jump).to_array(), rtol=1e-10)
# The first heated frame starts on the saturation isobar at the cold liquid volume.
cold_volume, cold_frame_temperature, _, _ = stages[0]
np.testing.assert_allclose((pressure(cold_volume, cold_frame_temperature) - saturation_pressure).to_array(), 0, atol=1e-10)
# Implicitly differentiating mismatch via its Gibbs-Duhem Jacobian recovers Clapeyron's relation without re-solving.
thermal_mismatch = (
    mv.l.dual() * (1 / (liquid - exclusion) - 1 / (vapour - exclusion))
    + mv.g.dual() * (entropy_jump + liquid / (liquid - exclusion) - vapour / (vapour - exclusion))
)
state_rate = -mismatch_jacobian(saturation_state, boiling_temperature).solve(thermal_mismatch)
saturation_slope = 1 / (liquid - exclusion) + pressure_slope(liquid, boiling_temperature) * liquid_volume(state_rate)
np.testing.assert_allclose(
    saturation_slope.to_array(), (latent_heat / (boiling_temperature * (vapour - liquid))).to_array(), rtol=1e-10
)